# IntelliGrade-H — TrOCR Fine-Tuning

### Before running:
1. `Runtime → Change runtime type → T4 GPU`
2. Make sure `datasets/handwriting/` is uploaded to Google Drive at:
   `My Drive/Intelligrade/datasets/handwriting/`
   (must contain `train/`, `val/`, `test/` — each with `images/` and `labels.txt`)
3. Run all cells top to bottom

### Expected time on T4 GPU
- IAM + CVL combined (~85k train samples, 10 epochs): ~60–75 minutes

## Cell 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')
    raise SystemExit('GPU required')

❌ No GPU — go to Runtime → Change runtime type → T4 GPU


SystemExit: GPU required

## Cell 2 — Mount Google Drive and set paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Dataset lives directly on Drive — no copying ──────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/Intelligrade'
TRAIN_DIR  = f'{DRIVE_BASE}/datasets/handwriting/train'
VAL_DIR    = f'{DRIVE_BASE}/datasets/handwriting/val'
TEST_DIR   = f'{DRIVE_BASE}/datasets/handwriting/test'
MODEL_OUT  = f'{DRIVE_BASE}/models/trocr-finetuned'
LOCAL_OUT  = '/content/trocr-finetuned'   # checkpoints saved here during training

os.makedirs(MODEL_OUT, exist_ok=True)
os.makedirs(LOCAL_OUT, exist_ok=True)

# Verify all splits exist
all_ok = True
for split, path in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    lf = os.path.join(path, 'labels.txt')
    if os.path.exists(lf):
        n = sum(1 for l in open(lf) if '\t' in l)
        print(f'  ✅ {split:6s}: {n:,} samples  →  {path}')
    else:
        print(f'  ❌ {split:6s}: labels.txt NOT found at {path}')
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Fix missing splits before continuing.')

print(f'\n✅ All splits found. Training from Drive directly.')
print(f'   Model will be saved to: {MODEL_OUT}')

Mounted at /content/drive
  ✅ train : 26,593 samples  →  /content/drive/MyDrive/Intelligrade/datasets/handwriting/train
  ✅ val   : 3,324 samples  →  /content/drive/MyDrive/Intelligrade/datasets/handwriting/val
  ✅ test  : 3,325 samples  →  /content/drive/MyDrive/Intelligrade/datasets/handwriting/test

✅ All splits found. Training from Drive directly.
   Model will be saved to: /content/drive/MyDrive/Intelligrade/models/trocr-finetuned


## Cell 3 — Install packages

In [ ]:
!pip uninstall -y transformers peft accelerate tokenizers -q
!pip install -q transformers==4.38.2 accelerate==0.27.2 tokenizers==0.15.2 evaluate==0.4.1 jiwer sentencepiece
import transformers, accelerate
print('transformers:', transformers.__version__)
print('accelerate  :', accelerate.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
transformers: 4.38.2
accelerate  : 0.27.2


## Cell 4 — Dataset class and metrics

In [ ]:
import torch
import numpy as np
import evaluate
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
from torch.utils.data import Dataset

processor = None  # assigned in Cell 6 after model load


class HandwritingDataset(Dataset):
    def __init__(self, data_dir, proc, max_length=128, augment=False):
        self.processor  = proc
        self.max_length = max_length
        self.augment    = augment
        self.data       = []
        data_dir   = Path(data_dir)
        images_dir = data_dir / 'images'
        with open(data_dir / 'labels.txt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if '\t' in line:
                    fname, text = line.split('\t', 1)
                    p = images_dir / fname
                    if p.exists():
                        self.data.append((str(p), text.strip()))
        print(f'  {data_dir.name:8s}: {len(self.data):,} samples')

    def __len__(self):
        return len(self.data)

    def _augment(self, image):
        import random
        if random.random() > 0.5:
            image = image.rotate(random.uniform(-3, 3), expand=True, fillcolor=255)
        if random.random() > 0.5:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
        if random.random() > 0.5:
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.2))
        if random.random() > 0.7:
            image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
        return image

    def __getitem__(self, idx):
        img_path, text = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.augment:
            image = self._augment(image)
        pixel_values = self.processor(image, return_tensors='pt').pixel_values.squeeze()
        labels = self.processor.tokenizer(
            text, padding='max_length', max_length=self.max_length,
            truncation=True, return_tensors='pt'
        ).input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}


def compute_metrics(eval_pred):
    preds, refs = eval_pred

    # Clamp to valid token range to prevent OverflowError from fp16
    vocab_size = processor.tokenizer.vocab_size
    preds = np.clip(preds, 0, vocab_size - 1).astype(np.int32)
    refs  = np.where(refs != -100, refs, processor.tokenizer.pad_token_id)
    refs  = np.clip(refs,  0, vocab_size - 1).astype(np.int32)

    pred_str = processor.batch_decode(preds, skip_special_tokens=True)
    ref_str  = processor.batch_decode(refs,  skip_special_tokens=True)

    cer = evaluate.load('cer').compute(predictions=pred_str, references=ref_str)
    wer = evaluate.load('wer').compute(predictions=pred_str, references=ref_str)
    return {'cer': cer, 'wer': wer}


print('✅ Dataset class and metrics ready')

✅ Dataset class and metrics ready


## Cell 5 — Training configuration
Only change these if needed.

In [ ]:
MODEL_NAME    = 'microsoft/trocr-small-handwritten'
EPOCHS        = 10
BATCH_SIZE    = 16    # reduce to 8 if you get CUDA out-of-memory
LEARNING_RATE = 5e-5
WARMUP_STEPS  = 500
GRAD_ACCUM    = 2     # effective batch size = 16 x 2 = 32
AUGMENT       = True

print(f'Model         : {MODEL_NAME}')
print(f'Epochs        : {EPOCHS}')
print(f'Batch size    : {BATCH_SIZE}  (effective: {BATCH_SIZE * GRAD_ACCUM})')
print(f'Learning rate : {LEARNING_RATE}')
print(f'Augmentation  : {AUGMENT}')
print(f'GPU           : {torch.cuda.get_device_name(0)}')

Model         : microsoft/trocr-small-handwritten
Epochs        : 10
Batch size    : 16  (effective: 32)
Learning rate : 5e-05
Augmentation  : True


AssertionError: Torch not compiled with CUDA enabled

## Cell 6 — Load model and train
Logs CER and WER after each epoch. Early stopping halts if val CER stops improving for 3 epochs.

In [ ]:
import shutil
from pathlib import Path

src = Path('/content/drive/MyDrive/Intelligrade/datasets/handwriting')
dst = Path('/content/datasets/handwriting')

if dst.exists():
    shutil.rmtree(dst)

print('Copying to local SSD (~3-5 mins)...')
shutil.copytree(src, dst)
print('✅ Copy done')

TRAIN_DIR = '/content/datasets/handwriting/train'
VAL_DIR   = '/content/datasets/handwriting/val'
TEST_DIR  = '/content/datasets/handwriting/test'

for split in ['train', 'val', 'test']:
    n = sum(1 for l in open(dst / split / 'labels.txt') if '\t' in l)
    print(f'  {split:6s}: {n:,} samples')

Copying to local SSD (~3-5 mins)...


In [ ]:
import os
from pathlib import Path
from transformers import (
    TrOCRProcessor, VisionEncoderDecoderModel,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    default_data_collator, EarlyStoppingCallback
)

# ── Set paths to local SSD ────────────────────────────────────────────────
TRAIN_DIR = '/content/datasets/handwriting/train'
VAL_DIR   = '/content/datasets/handwriting/val'
TEST_DIR  = '/content/datasets/handwriting/test'
LOCAL_OUT = '/content/trocr-finetuned'
MODEL_OUT = '/content/drive/MyDrive/Intelligrade/models/trocr-finetuned'

os.makedirs(LOCAL_OUT, exist_ok=True)
os.makedirs(MODEL_OUT, exist_ok=True)

# Verify splits
for split, path in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    lf = Path(path) / 'labels.txt'
    if lf.exists():
        n = sum(1 for l in open(lf) if '\t' in l)
        print(f'  ✅ {split:6s}: {n:,} samples')
    else:
        raise FileNotFoundError(f'{split} not found at {path} — copy may still be running')

# ── Load model ────────────────────────────────────────────────────────────
print('\nLoading model...')
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size
model.gradient_checkpointing_enable()
print('✅ Model loaded\n')

# ── Load datasets ────────────────────────────────────────────────────────-
print('Loading datasets from local SSD...')
train_ds = HandwritingDataset(TRAIN_DIR, processor, augment=AUGMENT)
val_ds   = HandwritingDataset(VAL_DIR,   processor, augment=False)

# ── Train ────────────────────────────────────────────────────────────────-
training_args = Seq2SeqTrainingArguments(
    output_dir                  = LOCAL_OUT,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    weight_decay                = 0.01,
    warmup_steps                = WARMUP_STEPS,
    num_train_epochs            = EPOCHS,
    predict_with_generate       = True,
    fp16                        = True,
    logging_steps               = 100,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'cer',
    greater_is_better           = False,
    remove_unused_columns       = False,
    dataloader_num_workers      = 2,
    report_to                   = [],
    generation_max_length       = 128 # Added to address UserWarning
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    data_collator   = default_data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print('\n🚀 Training started...')
trainer.train()


## Cell 7 — Evaluate on test set

In [ ]:
import numpy as np
from torch.utils.data import DataLoader

device      = 'cuda' if torch.cuda.is_available() else 'cpu'
test_ds     = HandwritingDataset(TEST_DIR, processor, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

cer_metric = evaluate.load('cer')
wer_metric = evaluate.load('wer')
all_preds, all_refs = [], []

model.to(device)
model.eval()

with torch.no_grad():
    for batch in test_loader:
        px      = batch['pixel_values'].to(device)
        gen_ids = model.generate(px, max_length=128, num_beams=4, early_stopping=True)
        preds   = processor.batch_decode(gen_ids, skip_special_tokens=True)
        labels  = batch['labels'].cpu().numpy()
        labels  = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
        refs    = processor.batch_decode(labels, skip_special_tokens=True)
        all_preds.extend(preds)
        all_refs.extend(refs)

cer       = cer_metric.compute(predictions=all_preds, references=all_refs)
wer       = wer_metric.compute(predictions=all_preds, references=all_refs)
total_c   = sum(len(r) for r in all_refs)
correct_c = sum(sum(p == r for p, r in zip(pred, ref))
                for pred, ref in zip(all_preds, all_refs))

print('\n' + '='*50)
print('📊 Test Set Results')
print('='*50)
print(f'Character Error Rate  : {cer:.4f}')
print(f'Word Error Rate       : {wer:.4f}')
print(f'Character Accuracy    : {(correct_c / total_c) * 100:.2f}%')
print(f'Samples evaluated     : {len(test_ds):,}')
print('='*50)
print('\nTarget: CER < 0.12 = Good   CER < 0.08 = Excellent')

## Cell 8 — Save model to Google Drive

In [ ]:
import shutil
from pathlib import Path

trainer.save_model(LOCAL_OUT)
processor.save_pretrained(LOCAL_OUT)

print(f'Copying model to Drive: {MODEL_OUT}')
if Path(MODEL_OUT).exists():
    shutil.rmtree(MODEL_OUT)
shutil.copytree(LOCAL_OUT, MODEL_OUT)

print('\n✅ Model saved to Google Drive!')
print(f'   {MODEL_OUT}')
print('\n── Deploy to IntelliGrade-H ──────────────────────────')
print('1. Right-click folder in Drive → Download as zip')
print('   My Drive/Intelligrade/models/trocr-finetuned/')
print('2. Extract to:')
print('   D:\\PycharmProjects\\IntelliGrade-H\\models\\trocr-finetuned\\')
print('3. Update .env:')
print('   TROCR_MODEL_PATH=models/trocr-finetuned')
print('4. python run.py')

## Cell 9 — Quick visual test (optional)
Picks a random image from the val set and shows prediction vs ground truth.

In [ ]:
import random
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

samples = [l.strip().split('\t') for l in open(Path(VAL_DIR) / 'labels.txt') if '\t' in l]
fname, ground_truth = random.choice(samples[:50])

img = Image.open(Path(VAL_DIR) / 'images' / fname).convert('RGB')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval()
px = processor(img, return_tensors='pt').pixel_values.to(device)

with torch.no_grad():
    ids  = model.generate(px, max_length=128, num_beams=4)
pred = processor.batch_decode(ids, skip_special_tokens=True)[0]

match = pred.strip().lower() == ground_truth.strip().lower()
print(f'Ground truth : {ground_truth}')
print(f'Prediction   : {pred}')
print(f'Match        : {"✅ Yes" if match else "❌ No"}')

plt.figure(figsize=(8, 2))
plt.imshow(img, cmap='gray')
plt.title(f'GT: "{ground_truth}"   Pred: "{pred}"', fontsize=10)
plt.axis('off')
plt.tight_layout()
plt.show()